<a href="https://colab.research.google.com/github/Divyaa2601/Predictive-Student-Success-Monitoring-System/blob/main/notebooks/03_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predictive Student Success Monitoring System
## Data Preprocessing and Feature Engineering

### Objective

This notebook prepares the synthetic student dataset for machine learning model development.

The preprocessing stage includes selecting appropriate predictive features, preventing target leakage, handling missing values, engineering academic performance trends, encoding the target variable, and splitting the dataset into training and testing sets.

The final processed data will be used to train and compare multiple machine learning algorithms for academic risk prediction.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
url = (
    "https://raw.githubusercontent.com/"
    "Divyaa2601/"
    "Predictive-Student-Success-Monitoring-System/"
    "main/data/student_success_dataset.csv"
)

df = pd.read_csv(url)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (2000, 15)


,student_id,semester,previous_gpa,attendance_pct,internal_1,internal_2,assignment_avg,assignment_completion_pct,quiz_avg,study_hours_weekly,class_participation,late_submissions,final_score,risk_level,edge_case_type
0,STU0001,4,6.13,64.9,60.5,56.8,62.3,66.0,51.3,10.4,6.1,1,41.5,High,normal
1,STU0002,4,7.19,52.5,55.7,66.4,68.5,67.7,45.2,5.1,2.7,6,58.9,Medium,normal
2,STU0003,5,6.40,52.1,53.8,55.9,63.4,65.4,32.3,12.0,2.3,0,48.4,Medium,normal
3,STU0004,6,6.86,74.9,64.1,42.2,77.8,52.2,52.7,10.5,4.8,5,54.5,Medium,normal
4,STU0005,4,9.26,74.4,82.3,64.9,78.2,72.3,71.1,9.9,9.3,4,70.7,Low,normal


In [3]:
df["internal_change"] = (
    df["internal_2"] - df["internal_1"]
)

df[
    ["internal_1", "internal_2", "internal_change"]
].head()

,internal_1,internal_2,internal_change
0,60.5,56.8,-3.7
1,55.7,66.4,10.7
2,53.8,55.9,2.1
3,64.1,42.2,-21.9
4,82.3,64.9,-17.4


In [4]:
features = [
    "semester",
    "previous_gpa",
    "attendance_pct",
    "internal_1",
    "internal_2",
    "assignment_avg",
    "assignment_completion_pct",
    "quiz_avg",
    "study_hours_weekly",
    "class_participation",
    "late_submissions",
    "internal_change"
]

X = df[features]

y = df["risk_level"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
for feature in features:
    print("-", feature)

Feature matrix shape: (2000, 12)
Target shape: (2000,)

Features:
- semester
- previous_gpa
- attendance_pct
- internal_1
- internal_2
- assignment_avg
- assignment_completion_pct
- quiz_avg
- study_hours_weekly
- class_participation
- late_submissions
- internal_change


### Feature Exclusion

Three variables are excluded from the machine learning input:

- `student_id` is only an identifier and does not contain predictive academic information.
- `final_score` is excluded because `risk_level` was derived from final score. Including it would cause target leakage and artificially inflate model performance.
- `edge_case_type` is metadata used to evaluate deliberately generated unusual student scenarios and is not information that would be available to the prediction system in a real deployment.

The machine learning models therefore use only academic and engagement information that could reasonably be available during the semester.

In [5]:
print("Missing values before preprocessing:\n")
print(X.isnull().sum())

Missing values before preprocessing:

semester                      0
previous_gpa                  0
attendance_pct                0
internal_1                    0
internal_2                    0
assignment_avg                0
assignment_completion_pct     0
quiz_avg                     60
study_hours_weekly           60
class_participation          60
late_submissions              0
internal_change               0
dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (1600, 12)
Testing features: (400, 12)

Training target distribution:
risk_level
Low       959
Medium    400
High      241
Name: count, dtype: int64

Testing target distribution:
risk_level
Low       240
Medium    100
High       60
Name: count, dtype: int64


In [7]:
print("Training distribution (%):")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting distribution (%):")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training distribution (%):
risk_level
Low       59.94
Medium    25.00
High      15.06
Name: proportion, dtype: float64

Testing distribution (%):
risk_level
Low       60.0
Medium    25.0
High      15.0
Name: proportion, dtype: float64


In [8]:
print("Missing values in training data:")
print(X_train.isnull().sum())

print("\nMissing values in testing data:")
print(X_test.isnull().sum())

Missing values in training data:
semester                      0
previous_gpa                  0
attendance_pct                0
internal_1                    0
internal_2                    0
assignment_avg                0
assignment_completion_pct     0
quiz_avg                     49
study_hours_weekly           48
class_participation          49
late_submissions              0
internal_change               0
dtype: int64

Missing values in testing data:
semester                      0
previous_gpa                  0
attendance_pct                0
internal_1                    0
internal_2                    0
assignment_avg                0
assignment_completion_pct     0
quiz_avg                     11
study_hours_weekly           12
class_participation          11
late_submissions              0
internal_change               0
dtype: int64


In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing values after imputation:")

print("\nTraining:")
print(X_train_imputed.isnull().sum())

print("\nTesting:")
print(X_test_imputed.isnull().sum())

Missing values after imputation:

Training:
semester                     0
previous_gpa                 0
attendance_pct               0
internal_1                   0
internal_2                   0
assignment_avg               0
assignment_completion_pct    0
quiz_avg                     0
study_hours_weekly           0
class_participation          0
late_submissions             0
internal_change              0
dtype: int64

Testing:
semester                     0
previous_gpa                 0
attendance_pct               0
internal_1                   0
internal_2                   0
assignment_avg               0
assignment_completion_pct    0
quiz_avg                     0
study_hours_weekly           0
class_participation          0
late_submissions             0
internal_change              0
dtype: int64


In [10]:
imputation_values = pd.Series(
    imputer.statistics_,
    index=X_train.columns
)

print("Median values learned from training data:")
print(imputation_values)

Median values learned from training data:
semester                      5.00
previous_gpa                  7.18
attendance_pct               74.40
internal_1                   63.80
internal_2                   61.60
assignment_avg               65.90
assignment_completion_pct    70.10
quiz_avg                     57.70
study_hours_weekly           11.60
class_participation           5.40
late_submissions              3.00
internal_change              -1.70
dtype: float64


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imputed),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling completed successfully.")

Scaling completed successfully.


In [12]:
X_train_scaled.head()

,semester,previous_gpa,attendance_pct,internal_1,internal_2,assignment_avg,assignment_completion_pct,quiz_avg,study_hours_weekly,class_participation,late_submissions,internal_change
800,-1.351557,-0.917597,-0.874526,0.450240,-0.416811,0.258172,-0.718577,0.070597,-0.643268,0.893133,1.196729,-1.088710
496,-0.465290,0.201213,0.573038,1.621375,-0.751811,-0.567627,-0.111808,0.472912,0.465121,-0.332910,1.196729,-2.927397
656,0.420977,-0.169477,-0.458103,-0.624745,-0.567229,0.602930,-1.182577,-0.896004,-1.352638,-2.128186,0.739853,-0.008037
1616,0.420977,-0.546907,-2.639364,0.061300,0.082965,-0.617406,-3.062372,0.002674,-0.953617,-0.508058,2.567357,0.036991
688,-1.351557,-2.103806,0.447450,-0.765196,-1.780278,-1.328812,-1.004116,-1.564788,0.309947,-0.858356,2.110481,-1.448934


In [13]:
print("Mean after scaling:")
print(X_train_scaled.mean().round(2))

print("\nStandard deviation after scaling:")
print(X_train_scaled.std().round(2))

Mean after scaling:
semester                    -0.0
previous_gpa                -0.0
attendance_pct               0.0
internal_1                  -0.0
internal_2                  -0.0
assignment_avg               0.0
assignment_completion_pct   -0.0
quiz_avg                    -0.0
study_hours_weekly          -0.0
class_participation         -0.0
late_submissions            -0.0
internal_change             -0.0
dtype: float64

Standard deviation after scaling:
semester                     1.0
previous_gpa                 1.0
attendance_pct               1.0
internal_1                   1.0
internal_2                   1.0
assignment_avg               1.0
assignment_completion_pct    1.0
quiz_avg                     1.0
study_hours_weekly           1.0
class_participation          1.0
late_submissions             1.0
internal_change              1.0
dtype: float64


In [14]:
print("PREPROCESSING SUMMARY")
print("-" * 40)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Number of features:", X_train.shape[1])

print("\nMissing values after imputation:")
print(
    "Training:",
    X_train_imputed.isnull().sum().sum()
)
print(
    "Testing:",
    X_test_imputed.isnull().sum().sum()
)

print("\nScaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nTarget classes:")
print(y_train.value_counts())

PREPROCESSING SUMMARY
----------------------------------------
Training samples: 1600
Testing samples: 400
Number of features: 12

Missing values after imputation:
Training: 0
Testing: 0

Scaled training shape: (1600, 12)
Scaled testing shape: (400, 12)

Target classes:
risk_level
Low       959
Medium    400
High      241
Name: count, dtype: int64


## Preprocessing Summary

The dataset was divided into 80% training data and 20% testing data using
stratified sampling to preserve the distribution of Low, Medium, and High
academic risk categories.

Missing values in the selected academic and engagement features were handled
using median imputation. The imputer was fitted only on the training dataset
to prevent information from the test dataset from influencing preprocessing.

An additional feature, `internal_change`, was created to represent the change
between Internal Assessment 1 and Internal Assessment 2.

Standardization was applied for algorithms that are sensitive to feature
scales, such as Logistic Regression. The original imputed feature values are
retained for tree-based algorithms such as Decision Tree, Random Forest, and
XGBoost.

The variables `student_id`, `final_score`, and `edge_case_type` were excluded
from the predictive feature set to prevent irrelevant information and target
leakage.

The processed dataset is now ready for machine learning model development and
evaluation.